# Fine-Tuning

This notebook now defines the config, device helpers, and fine-tuner class inline so the QLoRA setup stays self-contained.

## 1. Install dependencies

Unsloth's installer is version-sensitive to the Colab CUDA/torch build, so we use their official Colab-safe install command rather than plain `pip install unsloth`.

In [ ]:
import os, sys, logging, random
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import pandas as pd


def get_logger(name: str = "ecommerce_llm", level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(level)
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)
        formatter = logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            datefmt="%H:%M:%S",
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    return logger


logger = get_logger("notebook05")


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np
        import torch
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


@dataclass
class Config:
    project_root: Path = field(
        default_factory=lambda: Path(os.environ.get("PROJECT_ROOT", Path.cwd().parent)).resolve()
    )
    seed: int = 42
    hf_dataset_name: str = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
    hf_dataset_config: Optional[str] = None
    train_split_ratio: float = 0.8
    val_split_ratio: float = 0.1
    test_split_ratio: float = 0.1
    base_model_name: str = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
    max_seq_length: int = 2048
    load_in_4bit: bool = True
    lora_r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0
    lora_target_modules: tuple = (
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    )
    use_gradient_checkpointing: str = "unsloth"
    output_dir: str = "outputs"
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    learning_rate: float = 2e-4
    lr_scheduler_type: str = "linear"
    warmup_ratio: float = 0.05
    weight_decay: float = 0.01
    logging_steps: int = 10
    eval_strategy: str = "steps"
    eval_steps: int = 50
    save_strategy: str = "steps"
    save_steps: int = 50
    save_total_limit: int = 2
    bf16: bool = True
    fp16: bool = False
    optim: str = "adamw_8bit"
    report_to: str = "none"
    system_prompt: str = (
        "You are a helpful, polite Ecommerce Customer Support Assistant. "
        "You answer questions about orders, shipping, refunds, returns, "
        "payments, coupons, delivery, and account management. "
        "If a question is unrelated to ecommerce customer support, politely "
        "say that you can only help with ecommerce support questions."
    )
    instruction_header: str = "### Instruction:"
    response_header: str = "### Response:"
    eval_num_samples: int = 50
    eval_max_new_tokens: int = 128
    default_temperature: float = 0.7
    default_max_new_tokens: int = 256
    default_top_p: float = 0.9

    def __post_init__(self) -> None:
        set_seed(self.seed)

    @property
    def data_dir(self) -> Path:
        return self.project_root / "data"

    @property
    def raw_data_dir(self) -> Path:
        return self.data_dir / "raw"

    @property
    def processed_data_dir(self) -> Path:
        return self.data_dir / "processed"

    @property
    def sample_data_dir(self) -> Path:
        return self.data_dir / "sample"

    @property
    def models_dir(self) -> Path:
        return self.project_root / "models"

    @property
    def base_model_dir(self) -> Path:
        return self.models_dir / "base_model"

    @property
    def lora_adapter_dir(self) -> Path:
        return self.models_dir / "lora_adapter"

    @property
    def merged_model_dir(self) -> Path:
        return self.models_dir / "merged_model"

    def ensure_dirs(self) -> None:
        for path in (
            self.raw_data_dir,
            self.processed_data_dir,
            self.sample_data_dir,
            self.base_model_dir,
            self.lora_adapter_dir,
            self.merged_model_dir,
        ):
            path.mkdir(parents=True, exist_ok=True)


def load_jsonl(path) -> list[dict]:
    import json
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def detect_device() -> str:
    try:
        import torch
        if torch.cuda.is_available():
            name = torch.cuda.get_device_name(0)
            logger.info(f"CUDA GPU detected: {name}")
            return "cuda"
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            logger.info("Apple MPS backend detected.")
            return "mps"
    except ImportError:
        logger.warning("PyTorch not installed yet; defaulting device check to CPU.")
    logger.warning("No GPU detected — falling back to CPU (training will be slow).")
    return "cpu"


def supports_bf16() -> bool:
    try:
        import torch
        return torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    except Exception:
        return False


class EcommerceFineTuner:
    def __init__(self, config: Optional[Config] = None):
        self.config = config or Config()
        self.model = None
        self.tokenizer = None
        self.trainer = None

    def load_base_model(self):
        from unsloth import FastLanguageModel

        cfg = self.config
        device = detect_device()
        if device != "cuda":
            logger.warning(
                "No CUDA GPU detected. 4-bit QLoRA training requires a GPU "
                "(Colab: Runtime > Change runtime type > T4 GPU)."
            )

        logger.info(f"Loading base model: {cfg.base_model_name}")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=cfg.base_model_name,
            max_seq_length=cfg.max_seq_length,
            load_in_4bit=cfg.load_in_4bit,
            dtype=None,
        )
        self.model, self.tokenizer = model, tokenizer
        return model, tokenizer

    def apply_lora(self):
        from unsloth import FastLanguageModel

        if self.model is None:
            raise RuntimeError("Call load_base_model() before apply_lora().")

        cfg = self.config
        logger.info(
            f"Applying LoRA: r={cfg.lora_r}, alpha={cfg.lora_alpha}, targets={cfg.lora_target_modules}"
        )
        self.model = FastLanguageModel.get_peft_model(
            self.model,
            r=cfg.lora_r,
            target_modules=list(cfg.lora_target_modules),
            lora_alpha=cfg.lora_alpha,
            lora_dropout=cfg.lora_dropout,
            bias="none",
            use_gradient_checkpointing=cfg.use_gradient_checkpointing,
            random_state=cfg.seed,
        )
        return self.model

    def build_trainer(self, train_dataset, eval_dataset=None):
        from trl import SFTTrainer, SFTConfig

        if self.model is None or self.tokenizer is None:
            raise RuntimeError("Call load_base_model() and apply_lora() first.")

        cfg = self.config
        use_bf16 = cfg.bf16 and supports_bf16()
        use_fp16 = not use_bf16

        sft_config = SFTConfig(
            output_dir=cfg.output_dir,
            num_train_epochs=cfg.num_train_epochs,
            per_device_train_batch_size=cfg.per_device_train_batch_size,
            per_device_eval_batch_size=cfg.per_device_eval_batch_size,
            gradient_accumulation_steps=cfg.gradient_accumulation_steps,
            learning_rate=cfg.learning_rate,
            lr_scheduler_type=cfg.lr_scheduler_type,
            warmup_ratio=cfg.warmup_ratio,
            weight_decay=cfg.weight_decay,
            logging_steps=cfg.logging_steps,
            eval_strategy=cfg.eval_strategy if eval_dataset is not None else "no",
            eval_steps=cfg.eval_steps if eval_dataset is not None else None,
            save_strategy=cfg.save_strategy,
            save_steps=cfg.save_steps,
            save_total_limit=cfg.save_total_limit,
            bf16=use_bf16,
            fp16=use_fp16,
            optim=cfg.optim,
            report_to=cfg.report_to,
            seed=cfg.seed,
            max_seq_length=cfg.max_seq_length,
            dataset_text_field="text",
            packing=False,
        )

        logger.info(
            f"Building SFTTrainer (bf16={use_bf16}, fp16={use_fp16}, epochs={cfg.num_train_epochs}, "
            f"effective_batch={cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps})"
        )

        self.trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            args=sft_config,
        )
        return self.trainer

    def train(self):
        if self.trainer is None:
            raise RuntimeError("Call build_trainer() before train().")
        logger.info("Starting training...")
        result = self.trainer.train()
        logger.info(f"Training finished. Final metrics: {result.metrics}")
        return result

    def save_adapter(self, save_dir: Optional[str] = None) -> Path:
        save_dir = Path(save_dir or self.config.lora_adapter_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        self.model.save_pretrained(str(save_dir))
        self.tokenizer.save_pretrained(str(save_dir))
        logger.info(f"Saved LoRA adapter + tokenizer -> {save_dir}")
        return save_dir

    def get_loss_history(self) -> "pd.DataFrame":
        if self.trainer is None:
            raise RuntimeError("No trainer available yet.")
        return pd.DataFrame(self.trainer.state.log_history)


cfg = Config()
cfg.ensure_dirs()

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
%%capture
# Colab-safe Unsloth install (handles the correct xformers/torch/CUDA combo)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes
!pip install -q datasets sentencepiece


## 2. GPU detection

In [ ]:
logger = get_logger("notebook05")
device = detect_device()
print(f"Device: {device}")
print(f"bf16 supported: {supports_bf16()}")

if device != "cuda":
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > Hardware "
        "accelerator > T4 GPU, then re-run this notebook from the top."
    )

!nvidia-smi


## 3. Load config and formatted datasets

In [ ]:
cfg = Config()
cfg.ensure_dirs()

train_records = load_jsonl(cfg.processed_data_dir / "train.jsonl")
val_records = load_jsonl(cfg.processed_data_dir / "val.jsonl")

train_dataset = Dataset.from_list(train_records)
val_dataset = Dataset.from_list(val_records)

print(train_dataset)
print(val_dataset)
print("\nSample training text:\n")
print(train_dataset[0]["text"])


## 4. Load base model in 4-bit (QLoRA) + attach LoRA adapter

In [ ]:
fine_tuner = EcommerceFineTuner(cfg)
model, tokenizer = fine_tuner.load_base_model()
model = fine_tuner.apply_lora()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")


## 5. Configure and run the SFTTrainer

Uses:
- `max_seq_length` from config
- gradient checkpointing (Unsloth's optimized variant)
- bf16 on Ampere+/T4-supported GPUs, fp16 fallback otherwise
- gradient accumulation for a larger effective batch size
- linear LR schedule with warmup
- periodic evaluation on the validation split

In [ ]:
trainer = fine_tuner.build_trainer(train_dataset, val_dataset)
print(trainer.args)


In [ ]:
train_result = fine_tuner.train()
train_result.metrics


## 6. Plot training & validation loss

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history = fine_tuner.get_loss_history()
train_hist = history[history["loss"].notna()][["step", "loss"]] if "step" in history.columns else history[history["loss"].notna()][["loss"]].reset_index()
eval_hist = history[history.get("eval_loss").notna()] if "eval_loss" in history.columns else pd.DataFrame()

plt.figure(figsize=(9, 5))
if "step" in history.columns:
    plt.plot(train_hist["step"], train_hist["loss"], label="train_loss", marker="o", markersize=3)
    if not eval_hist.empty:
        plt.plot(eval_hist["step"], eval_hist["eval_loss"], label="val_loss", marker="s", markersize=4)
else:
    plt.plot(train_hist["loss"].values, label="train_loss")

plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 7. Save the LoRA adapter + tokenizer

In [ ]:
save_dir = fine_tuner.save_adapter()
print(f"Adapter saved to: {save_dir}")
!ls -lh {save_dir}


## 8. Quick smoke test

Generate one response right away (before formal evaluation in notebook 06) to sanity-check the fine-tune worked.

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

test_instruction = "Where is my order? It's been 5 days."
prompt = f"{cfg.instruction_header}\nCustomer:\n{test_instruction}\n\n{cfg.response_header}\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=120, temperature=0.7, do_sample=True)
print(tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))


## Summary

- Loaded `Config.base_model_name` in 4-bit and attached a LoRA adapter (r=16, targeting attention + MLP projections)
- Trained with `SFTTrainer` using gradient accumulation, warmup, and bf16/fp16 auto-selection
- Plotted train/validation loss to confirm convergence
- Saved the lightweight LoRA adapter to `models/lora_adapter/`

Next: `06_evaluation.ipynb` for BLEU/ROUGE/perplexity scoring, then `07_inference.ipynb` to merge and deploy.